# HiRoute v3 Remote Runner

This notebook clones the GitHub repository, builds `ns-3`/`ndnSIM`, runs selected `smartcity_v3` experiment matrices, and pushes registries plus aggregate outputs back to git.

Use a runtime with high **system RAM**. ndnSIM runs are memory-bound on the host, not GPU-VRAM-bound.

In [ ]:
from getpass import getpass
import os
import pathlib
import subprocess

REPO_HTTPS = "https://github.com/<your-org-or-user>/HiRoute.git"
BRANCH = "main"
WORKDIR = pathlib.Path("/content/HiRoute")
GIT_USER_NAME = "HiRoute Colab Runner"
GIT_USER_EMAIL = "you@example.com"
PUSH_BRANCH = BRANCH
TOKEN = getpass("GitHub token with repo access: ")

clone_url = REPO_HTTPS.replace("https://", f"https://{TOKEN}@")
if WORKDIR.exists():
    subprocess.run(["rm", "-rf", str(WORKDIR)], check=True)
subprocess.run(["git", "clone", "--branch", BRANCH, clone_url, str(WORKDIR)], check=True)
subprocess.run(["git", "config", "user.name", GIT_USER_NAME], cwd=WORKDIR, check=True)
subprocess.run(["git", "config", "user.email", GIT_USER_EMAIL], cwd=WORKDIR, check=True)
subprocess.run(["git", "remote", "set-url", "origin", clone_url], cwd=WORKDIR, check=True)
print(WORKDIR)

In [ ]:
%%bash
set -eux
apt-get update
DEBIAN_FRONTEND=noninteractive apt-get install -y \
  build-essential g++ gcc python3-dev cmake pkg-config sqlite3 libsqlite3-dev \
  libboost-all-dev libgsl-dev git
cd /content/HiRoute
python3 -m venv .venv
.venv/bin/pip install --upgrade pip setuptools wheel
.venv/bin/pip install -r requirements.txt
cd ns-3
./waf configure --enable-examples --disable-python
./waf build

In [ ]:
%%bash
set -eux
cd /content/HiRoute
.venv/bin/python scripts/build_dataset/validate_dataset.py --config configs/datasets/smartcity_v3.yaml --topology-config configs/topologies/rocketfuel_3967_exodus.yaml
.venv/bin/python scripts/build_dataset/validate_dataset.py --config configs/datasets/smartcity_v3.yaml --topology-config configs/topologies/rocketfuel_1239_sprint.yaml

In [ ]:
MAX_WORKERS = 1
RUN_ROUTING_MAIN = True
RUN_OBJECT_MAIN = True
RUN_ABLATION = True
RUN_SCALING = False
RUN_ROBUSTNESS = False

In [ ]:
experiments = [
    (RUN_ROUTING_MAIN, "configs/experiments/exp_routing_main_v3.yaml"),
    (RUN_OBJECT_MAIN, "configs/experiments/exp_object_main_v3.yaml"),
    (RUN_ABLATION, "configs/experiments/exp_ablation_v3.yaml"),
    (RUN_SCALING, "configs/experiments/exp_scaling_v3.yaml"),
    (RUN_ROBUSTNESS, "configs/experiments/exp_robustness_v3.yaml"),
]

for enabled, experiment_path in experiments:
    if not enabled:
        continue
    cmd = [
        str(WORKDIR / ".venv/bin/python"),
        "scripts/run/run_experiment_matrix.py",
        "--experiment",
        experiment_path,
        "--max-workers",
        str(MAX_WORKERS),
        "--postprocess",
        "--validate",
    ]
    print("Running", " ".join(cmd))
    subprocess.run(cmd, cwd=WORKDIR, check=True)

In [ ]:
print(subprocess.run(["git", "status", "--short"], cwd=WORKDIR, check=True, capture_output=True, text=True).stdout)
add_targets = [
    "runs/registry/runs.csv",
    "runs/registry/promoted_runs.csv",
    "results/aggregate/v3",
    "results/tables/v3",
    "results/figures/v3",
]
existing_targets = [target for target in add_targets if (WORKDIR / target).exists()]
if existing_targets:
    subprocess.run(["git", "add", *existing_targets], cwd=WORKDIR, check=True)
commit = subprocess.run(
    ["git", "commit", "-m", "eval: colab rerun update"],
    cwd=WORKDIR,
    check=False,
    capture_output=True,
    text=True,
)
print(commit.stdout or commit.stderr)
subprocess.run(["git", "pull", "--rebase", "origin", PUSH_BRANCH], cwd=WORKDIR, check=True)
subprocess.run(["git", "push", "origin", f"HEAD:{PUSH_BRANCH}"], cwd=WORKDIR, check=True)

## Pull results back locally

On the local machine:

```bash
cd /Users/jiyuan/Desktop/HiRoute
git pull --rebase origin <branch>
```

Canonical return artifacts are `runs/registry/*.csv`, `results/aggregate/v3/*.csv`, `results/tables/v3/*.csv`, and `results/figures/v3/*.pdf`.